## Setup

In [2]:
from pyspark.sql import Row, SparkSession

spark = (
    SparkSession.builder.appName("sample_spark_session")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/28 15:18:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
from pathlib import Path
from generate_data import run
from run_ddl import run_ddl

run(scaling_factor=0.01, format="csv", output_folder_path="./data") # data will be nGB on your disk\n",
run_ddl(data_path=Path("./data"), spark=spark, recreate=True)

In [ ]:
spark.sql("use prod.db")
spark.sql("show tables").show()

#  Design Patterns: Well designed fact and dimension pipelines will take care of 90% of your job 

## Facts and dimensions are the most critical data products DE create [10 min]

[5 min]

![Standard Data Flow](../images/data-flow.png)

- raw -> bronze: tools: fivetran, kafka -> S3 connector, CDC, dlt, etc
- mart/gold: semantic layer, metrics layer, views, JOINS + GROUP BYs
- some companies use a inmon layer pre-kimball; usually at companies where there are mutliple organizations and there is a need to combien them into a unified data model
- most companies follow the multi hop architecture, it can take slightly different forms + different naming (bronze/silver/gold, raw/base/stage/clean, marts, etc)
- essentially the idea is
    1. Bring upstream data into a cloud store 
    2. Create a data type and naming convention confirming dataset of the raw data called bronze (some companies combine 1 & 2, some create 2 as views on top of 1)
    3. [Optional] Create a 3nf to unify data from multiple systems. This is not very common, but prevelant at companies where data team is expected to unify data from multiple organizations (think orgs that were acquired over time)
    4. The Kimball layer; Data is modelled as facts, dims and bridge tables. Commonly referred to as star schema. Highly denormalized, ie keep number of tables low by combining multiple upstream tables to a few dimension tables.
    5. well designed Facts and dimensions are able to answer any question about a business, given that the data is collected. However this often involves joining them and groupin them, while calculating the right metrics. The problem is that the SQL knowledge and metric computation knowledge are hard for most non technical people. So DEs often create marts/summary/pre-aggregated tables, which are basically joined + grouped versions of the raw fact and dim tables. There is also metric layers (lookml, etc) where DEs define the dims and metrics as code and users can drag-and-drop them to create dashboards. Lately there is also the semantic layer which aims to enable AI agents to write the correct queries in response to user questions.
- The messy middle, which gives DE the leverage to asymmetrically impact outcomes is the kimball layer. 
- The ease and reliability of using data depends on the kimball layer. 
- In this workshop we will go over the patterns to design your fact and dim pipelines. 
- By the end of this post you will have a flowchart that you can use to quickly define you fact and dim tables for your use case.
- add: image showing dump -> bronze -> silver -> gold , with highlight around bronze layer and tools/teams under the bronze and gold layers

[5 min]
Q & A

**Additional Links**

1. Fivetran 
2. CDC with debezium 
3. OSI semantic layer

TL;DR [5 min]

add: image representing the fact/dim design patterns

## The Fact Pipeline Design: Read input incrementally, Left Join mapping data & create new rows, INSERT OVERWRITE BY PARTITION to output [25 m]

- add: image of extracting orders data based on time, enriching orders table with mapping data (dim_date), and loading with insert overwrite to output 

![Standard Fact ETL](../images/etl-fact.png)

#### Exercise [5 min]

Use [fct_line_items.py](./fct_line_items.py) as a blueprint & create `fct_orders.py`. 

![Time range](../images/daily-run-time-range.png)

**Note**: We can also use `select * from source where event_ts > (select max(event_ts) from destination)`, although that will make maintenance harder, especially backfills and re-runs.

Then run `fct_orders.py` similar to how we run `fct_line_items.py` as shown below.

In [ ]:
! uv run ./fct_line_items.py --start-time 1995-01-01 --end-time 1996-01-01

In [ ]:
spark.sql("select min(o_orderdate), max(o_orderdate) from prod.db.orders").show()

In [ ]:
spark.sql("select min(full_date), max(full_date) from prod.db.dim_date").show()

In [ ]:
! uv run ./fct_orders.py --start-time 1995-01-01 --end-time 1996-01-01

In [ ]:
spark.table("local.silver.fct_orders").limit(2).toPandas()


[5 min]
- Most fact table involves enrichment, ie. COALESCE + LEFT JOIN, creating new column,
- However if the columns created encoded business logic, they should be pushed to the application layer
- for example: a column computing cost after discount is valid, as its using existing logic 
- however if you find yourself changing the discount % based on a product id, because `we didn't do it in the backend` you have to push the application team to implement it 
- date range [str & end)
- alt created_at > max(created_at) from output

[10 min]
- add: image of transformation types
- in addition to standard enrichement style transformation, there are common patterns where you will need to do window function or self join or fact-fact join, let's look at some example
- Sessionization/attributions: idea is to tag related rows. Note that event order plays a key role here. I.e. you can't attribute a conversion to a click if you converstion row arrives and click hasnt (note: late arriving data), typicaly the reason people wait a few hours before processing data. This pattern typically involves window functions, however in complex cases the
- add: code example
- deduplication: either caused how data is dumped into the bronze layer or how your system works. For example if a user accidentally double clicks on a purchase button do you record it as 2 purchases or 1 purchase?
- add: code example
- UNION: Typical when you buy same type of data from multiple vendors you will need to combine them to have the same structure. For example you purchase behaviour data from Facebook and google, and have to combine it (naive example). These are flaky, especially with constantly changing input structures, and will often involve columns that are present in one but absent in another. when faced with such as situation, look for industry standards and match your data to it. For example, [OpenRTB](https://github.com/InteractiveAdvertisingBureau/openrtb2.x/blob/main/2.6.md#3233---object-refresh-) is a common standard used by most ads companies.
- DIFF: Common when ingesting data from vendors. Typically you identify the diff rows and process them. You may wonder why not just process the entire data, this may be due to various reasons, such as the vendor only sends data from past n days, but you need to keep track of history. The vendor data is not good quality so you need to check that the changes make sense. Your system can't handle the amount of data to process if you didnt diff it.
- add: links for further reading; adv sql, sql 25 tips, 

### Example [5 min]

Write a transformation query to remove duplicates from the clickstream data (created as CTE below)

![De-duplication](../images/dupclick.png)

In [ ]:
spark.sql("""
WITH clickstream AS (
    SELECT
        1 AS user_id, '2024-07-01 10:00:00' AS click_time UNION ALL
    SELECT
        1 AS user_id, '2024-07-01 10:05:00' AS click_time UNION ALL
    SELECT
        1 AS user_id, '2024-07-01 10:10:00' AS click_time UNION ALL
    SELECT
        1 AS user_id, '2024-07-01 10:10:00' AS click_time UNION ALL
    SELECT
        1 AS user_id, '2024-07-01 10:10:00' AS click_time UNION ALL
    SELECT
        1 AS user_id, '2024-07-01 10:10:00' AS click_time UNION ALL
    SELECT
        2 AS user_id, '2024-07-01 10:15:00' AS click_time UNION ALL
    SELECT
        2 AS user_id, '2024-07-01 10:20:00' AS click_time UNION ALL
    SELECT
        2 AS user_id, '2024-07-01 10:20:00' AS click_time UNION ALL
    SELECT
        2 AS user_id, '2024-07-01 10:20:00' AS click_time UNION ALL
    SELECT
        2 AS user_id, '2024-07-01 10:20:00' AS click_time UNION ALL
    SELECT
        2 AS user_id, '2024-07-01 10:25:00' AS click_time
),
ranked_clicks AS (
    SELECT
        user_id,
        click_time,
        ROW_NUMBER() OVER (PARTITION BY user_id, click_time ORDER BY click_time) AS click_rank
    FROM
        clickstream
)
SELECT
    user_id,
    click_time,
    click_rank
FROM
    ranked_clicks
WHERE
    click_rank = 1
""").show()

ADD: feedback link and possible sign up for hands-on data engineering design pattern book (title: TBD)

## Dim pipeline design: select * read, inner join, overwrite table [10 m]

- most dim pipelines read full inputs as sources and inner join them to create denormalized tables

#### Example [10 min]

Run the scripts `dim_customer_v1.py` and `dim_customer_v2.py`. 

Answer the following questions

1. What do you think is the difference in results? And why does it happen?
2. Which approach would you choose to implement in a pipeline and why? What are the pros and cons of each approach for the user of your data?
3. Is there another approach, one that lets you use inner join, but not loose data? What would that look like?
                                                                                                                   
**Hint** Pay attention to the transformation logic

- the key things to be aware of here are the completeness of input data
- you can use a left join, however this will flow downstream and impact the fact tabes

In [ ]:
spark.sql("drop table if exists prod.db.dim_mktsegment");

spark.sql("""
    CREATE TABLE IF NOT EXISTS prod.db.dim_mktsegment (
      c_mktsegment        STRING,
      segment_description STRING,
      priority_tier       STRING
    ) USING iceberg
    TBLPROPERTIES (
      'format-version' = '2'
    )
    """);

spark.sql("""
    INSERT INTO prod.db.dim_mktsegment VALUES
      ('MACHINERY',  'Industrial machinery and equipment buyers', 'High'),
      ('AUTOMOBILE', 'Automotive and vehicle-related customers',   'High'),
      ('BUILDING',   'Construction and building materials sector', 'Medium'),
      ('HOUSEHOLD',  'Household goods and consumer products',      'Medium')
    """)

In [ ]:
! uv run ./dim_customer_v1.py

In [ ]:
! uv run ./dim_customer_v2.py

In [ ]:
spark.sql("select c_mktsegment, segment_description, count(*) from local.silver.dim_customer_v1 group by 1, 2")

In [ ]:
spark.sql("select c_mktsegment, segment_description, count(*) from local.silver.dim_customer_v2 group by 1, 2")

![Dimension Join Types](../images/dim-join-types.png)

In [ ]:
spark.sql("select c_mktsegment, count(*) from customer group by 1").show()

In [ ]:
spark.sql("""
select c.*
, s.segment_description
, s.priority_tier
from prod.db.customer c
join prod.db.dim_mktsegment s
on c.c_mktsegment = s.c_mktsegment
where c.c_mktsegment = 'FURNITURE'
""").show(5)

In [ ]:
spark.sql("""
with dim_customer as (select c.*
, s.segment_description
, s.priority_tier
from prod.db.customer c
join prod.db.dim_mktsegment s
on c.c_mktsegment = s.c_mktsegment
)
select c_mktsegment, segment_description, count(*) from dim_customer group by 1, 2
""").show(5)

* inner join lost us the `FURNITURE`
* this may be ok, but be mindful of the choice you make
* you canalso use COALESE + LEFT JOIN as shown below or decide to wait or decide that the data will cathch up in the next run
* if you chose to use inner join, you will need to ensure that the downstream users know that the data may not be complete. 
* add image showing options as diverging paths, and their cons

In [ ]:
spark.sql("""
with dim_customer as (select c.*
, COALESCE(s.segment_description, 'UNKNOWN') as segment_description
, COALESCE(s.priority_tier, 'UNKNOWN') as priority_tier
from prod.db.customer c
left join prod.db.dim_mktsegment s
on c.c_mktsegment = s.c_mktsegment
)
select c_mktsegment, segment_description, count(*) from dim_customer group by 1, 2
""").show(5)


#### exercise [5 min] run this pipeline, chek the output, what is wrong?  -- combine with previous
A: incomplete inner join
- explain: data completeness is an issue, you need to know when to run this pipeline (when inputs are complete) or wait for the next run to catch missing data. 


In [ ]:
spark.sql("drop table if exists prod.db.brand")
spark.sql("""
    CREATE TABLE IF NOT EXISTS prod.db.brand (
      brand_id    BIGINT,
      brand_name  STRING,
      country     STRING,
      created_at  TIMESTAMP,
      updated_at  TIMESTAMP
    ) USING iceberg
    TBLPROPERTIES (
      'format-version' = '2'
    )
    """)

spark.sql("drop table if exists prod.db.product")
spark.sql("""
    CREATE TABLE IF NOT EXISTS prod.db.product (
      product_id    BIGINT,
      product_name  STRING,
      brand_id      BIGINT,   -- FK -> brand.brand_id (1:1)
      price         DECIMAL(15,2),
      created_at    TIMESTAMP,
      updated_at    TIMESTAMP
    ) USING iceberg
    TBLPROPERTIES (
      'format-version' = '2'
    )
    """)

spark.sql("""
    INSERT INTO prod.db.brand VALUES
      (1, 'Acme',     'USA',     TIMESTAMP '2024-01-10 09:00:00', TIMESTAMP '2024-01-10 09:00:00'),
      (2, 'Globex',   'Germany', TIMESTAMP '2024-01-12 10:30:00', TIMESTAMP '2024-02-01 14:15:00'),
      (3, 'Initech',  'Canada',  TIMESTAMP '2024-01-15 11:45:00', TIMESTAMP '2024-01-15 11:45:00'),
      (4, 'Umbrella', 'Japan',   TIMESTAMP '2024-01-20 08:20:00', TIMESTAMP '2024-03-05 16:00:00')
    """)

spark.sql("""
    INSERT INTO prod.db.product VALUES
      (101, 'Cordless Drill',     1, 79.99,  TIMESTAMP '2024-01-11 09:00:00', TIMESTAMP '2024-01-14 12:00:00'),
      (102, 'Circular Saw',       1, 129.50, TIMESTAMP '2024-01-11 14:30:00', TIMESTAMP '2024-01-11 14:30:00'),
      (103, 'Smart Thermostat',   2, 199.00, TIMESTAMP '2024-01-13 10:00:00', TIMESTAMP '2024-01-18 16:20:00'),
      (104, 'LED Bulb 4-Pack',    2, 24.99,  TIMESTAMP '2024-01-14 11:15:00', TIMESTAMP '2024-01-14 11:15:00'),
      (105, 'Office Chair',       3, 149.95, TIMESTAMP '2024-01-16 08:45:00', TIMESTAMP '2024-01-19 13:00:00'),
      (106, 'Standing Desk',      3, 389.00, TIMESTAMP '2024-01-16 15:30:00', TIMESTAMP '2024-01-16 15:30:00'),
      (107, 'Air Purifier',       4, 159.99, TIMESTAMP '2024-01-20 09:10:00', TIMESTAMP '2024-01-20 09:10:00'),
      (108, 'Humidifier',         4, 59.99,  TIMESTAMP '2024-01-20 10:40:00', TIMESTAMP '2024-01-20 17:00:00'),
      (109, 'Espresso Machine',   1, 249.00, TIMESTAMP '2024-01-12 13:00:00', TIMESTAMP '2024-01-15 09:30:00'),
      (110, 'Robot Vacuum',       2, 299.99, TIMESTAMP '2024-01-15 13:30:00', TIMESTAMP '2024-01-17 10:45:00')
    """)

In [ ]:
spark.sql("""
    SELECT
      p.product_id,
      p.product_name,
      p.price,
      p.brand_id,
      b.brand_name,
      b.country,
      p.created_at  AS product_created_at,
      p.updated_at  AS product_updated_at,
      b.created_at  AS brand_created_at,
      b.updated_at  AS brand_updated_at
    FROM prod.db.product p
    JOIN prod.db.brand b
      ON p.brand_id = b.brand_id
    ORDER BY p.product_id
    """).show(2)

In [ ]:
spark.sql("select * from product where date(updated_at) = '2024-01-17' ").show()

In [ ]:
spark.sql("select * from brand where date(updated_at) = '2024-01-17' ").show()

In [ ]:
spark.sql("""
    with product_incremental as (select * from prod.db.product where date(updated_at) = '2024-01-17'),
    brand_incremental as (select * from prod.db.brand where date(updated_at) = '2024-01-17')
    SELECT
      p.product_id,
      p.product_name,
      p.price,
      p.brand_id,
      b.brand_name,
      b.country,
      p.created_at  AS product_created_at,
      p.updated_at  AS product_updated_at,
      b.created_at  AS brand_created_at,
      b.updated_at  AS brand_updated_at
    FROM product_incremental p
    JOIN brand_incremental b
      ON p.brand_id = b.brand_id
    ORDER BY p.product_id
    """).show(2)


#### exercise [10m] read this code, what is wrong? Why not incremental dim? 
- A: this chunks data points to other inputs old chunks data => missed data
- dim tables are often small (a few million rows) and full select is often the simplest to maintain
- add: image that shows inc dim + joins is tricky as a data now can point to an input from past, 


In [3]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS prod.db.customer_details (
      customer_id  STRING,
      name         STRING,
      email        STRING,
      created_at   DATE,
      updated_at   DATE
    ) USING iceberg
    TBLPROPERTIES (
      'format-version' = '2'
    )
    """)

spark.sql("""
    INSERT INTO prod.db.customer_details VALUES
      ('c1', 'customer_1_name', 'customer_1_email',    DATE '2026-01-01', DATE '2026-01-01'),
      ('c2', 'customer_2_name', 'customer_2_email',    DATE '2026-01-02', DATE '2026-01-02'),
      ('c3', 'customer_3_name', 'customer_3_email',    DATE '2026-01-02', DATE '2026-01-02'),
      ('c4', 'customer_4_name', 'customer_4_email',    DATE '2026-01-03', DATE '2026-01-03'),
      ('c1', 'customer_1_name', 'customer_1_email_v2', DATE '2026-01-01', DATE '2026-01-03')
    """)

spark.sql("""
    CREATE TABLE IF NOT EXISTS prod.db.customer_address (
      customer_id  STRING,
      address      STRING,
      created_at   DATE,
      updated_at   DATE
    ) USING iceberg
    TBLPROPERTIES (
      'format-version' = '2'
    )
    """)

spark.sql("""
    INSERT INTO prod.db.customer_address VALUES
      ('c1', 'customer_1_address', DATE '2026-01-01', DATE '2026-01-01'),
      ('c2', 'customer_2_address', DATE '2026-01-02', DATE '2026-01-02'),
      ('c3', 'customer_3_address', DATE '2026-01-02', DATE '2026-01-02'),
      ('c4', 'customer_4_address', DATE '2026-01-03', DATE '2026-01-03')
    """)

DataFrame[]

In [5]:
spark.sql("""
    SELECT
      c.customer_id,
      c.name,
      c.email,
      a.address,
      c.created_at,
      c.updated_at
    FROM prod.db.customer_details c
    JOIN prod.db.customer_address a
      ON c.customer_id = a.customer_id
    """).show()

+-----------+---------------+-------------------+------------------+----------+----------+
|customer_id|           name|              email|           address|created_at|updated_at|
+-----------+---------------+-------------------+------------------+----------+----------+
|         c1|customer_1_name|   customer_1_email|customer_1_address|2026-01-01|2026-01-01|
|         c2|customer_2_name|   customer_2_email|customer_2_address|2026-01-02|2026-01-02|
|         c3|customer_3_name|   customer_3_email|customer_3_address|2026-01-02|2026-01-02|
|         c4|customer_4_name|   customer_4_email|customer_4_address|2026-01-03|2026-01-03|
|         c1|customer_1_name|customer_1_email_v2|customer_1_address|2026-01-01|2026-01-03|
+-----------+---------------+-------------------+------------------+----------+----------+



In [6]:
spark.sql("""
    with customer_details_incremental as (select * from prod.db.customer_details where date(updated_at) = '2026-01-03'),
    customer_address_incremental as (select * from prod.db.customer_address where date(updated_at) = '2026-01-03')
    SELECT
      c.customer_id,
      c.name,
      c.email,
      a.address,
      c.created_at  AS customer_created_at,
      c.updated_at  AS customer_updated_at,
      a.created_at  AS address_created_at,
      a.updated_at  AS address_updated_at
    FROM customer_details_incremental c
    JOIN customer_address_incremental a
      ON c.customer_id = a.customer_id
    ORDER BY c.customer_id
    """).show()

+-----------+---------------+----------------+------------------+-------------------+-------------------+------------------+------------------+
|customer_id|           name|           email|           address|customer_created_at|customer_updated_at|address_created_at|address_updated_at|
+-----------+---------------+----------------+------------------+-------------------+-------------------+------------------+------------------+
|         c4|customer_4_name|customer_4_email|customer_4_address|         2026-01-03|         2026-01-03|        2026-01-03|        2026-01-03|
+-----------+---------------+----------------+------------------+-------------------+-------------------+------------------+------------------+



In [7]:
spark.sql("""
    with customer_details_incremental as (select * from prod.db.customer_details where date(updated_at) = '2026-01-03'),
    customer_address_incremental as (select * from prod.db.customer_address where date(updated_at) = '2026-01-03')
    SELECT
      c.customer_id,
      c.name,
      c.email,
      a.address,
      c.created_at  AS customer_created_at,
      c.updated_at  AS customer_updated_at,
      a.created_at  AS address_created_at,
      a.updated_at  AS address_updated_at
    FROM customer_details_incremental c
    LEFT JOIN customer_address_incremental a
      ON c.customer_id = a.customer_id
    ORDER BY c.customer_id
    """).show()

+-----------+---------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+
|customer_id|           name|              email|           address|customer_created_at|customer_updated_at|address_created_at|address_updated_at|
+-----------+---------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+
|         c1|customer_1_name|customer_1_email_v2|              NULL|         2026-01-01|         2026-01-03|              NULL|              NULL|
|         c4|customer_4_name|   customer_4_email|customer_4_address|         2026-01-03|         2026-01-03|        2026-01-03|        2026-01-03|
+-----------+---------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+



In [9]:
spark.sql("""
select * from prod.db.customer_address where customer_id = 'c1'
""").show()

+-----------+------------------+----------+----------+
|customer_id|           address|created_at|updated_at|
+-----------+------------------+----------+----------+
|         c1|customer_1_address|2026-01-01|2026-01-01|
+-----------+------------------+----------+----------+



![Incremental Dim Pipeline Problems](../images/inc-dim-inner-join-data-loss.png)

Note:
- data asset driven pipeline to ensure data completeness add:link
- scd2: IFF users require history. Most users don't know how to use scd2 or care. Be mindful SCD2's are deceptively complext. add screenshot example

## Highlevel next steps

* failure analysis: missing data, bad schema, rerun on fail, defensive programming, input validation, scale (reduce data shuffle & pratition prune)
 

## Recap

* add: table image
* share link to article 

In [ ]:
import pandas as pd
from great_tables import GT, md, style, loc

df = pd.DataFrame({
    "type": ["Fact", "Dim"],
    "extract": [
        "`SELECT * FROM source WHERE inserted_at >= start_time AND inserted_at < end_time`",
        "`SELECT * FROM source tables`",
    ],
    "transform": [
        "**Standard:** COALESCE + LEFT JOIN mapping tables<br>"
        "**Advanced:** deduplication, sessionization, UNION, DIFF with window functions and cross joins",
        "**Pattern 1:** driver table + INNER JOIN other tables → unknown, incomplete but fast<br>"
        "**Pattern 2:** driver table + LEFT JOIN other tables → known, incomplete; wait for input table to be complete → slow",
    ],
    "quality": [
        "* `COUNT(*)` in transformed data vs `COUNT(*)` in source",
        "* Unique / PK constraint \n * `COUNT(*)` compared to source",
    ],
    "load": [
        "Insert overwrite partition",
        "Overwrite full table (for SCD2 use `MERGE INTO`)",
    ],
    "opt_storage": [
        "Partition by `day(created_at)`",
        "Not needed (data small &lt; 10mil)",
    ],
    "opt_code": [
        "Broadcast join mapping tables.\n For window functions and self joins, aim to reduce data shuffle and use [SPJ](https://spark.apache.org/docs/latest/sql-performance-tuning.html#storage-partition-join)",
        "Not needed (data small &lt; 10mil)",
    ],
})

gt = (
    GT(df, rowname_col="type")
    .tab_header(
        title=md("**Design Patterns: Fact & Dimension Tables**"),
        subtitle="Reference for extract, transform, quality, load, and optimization stages",
    )
    .fmt_markdown(columns=["extract", "transform", "quality", "load", "opt_storage", "opt_code"])
    .cols_label(
        extract="Extract",
        transform="Transform",
        quality="Quality Check",
        load="Load",
        opt_storage="Storage",
        opt_code="Code",
    )
    .tab_spanner(label="Optimization", columns=["opt_storage", "opt_code"])
    .tab_stubhead(label="Table Type")
    .cols_align(align="left")
    .cols_width({
        "type": "70px",
        "extract": "190px",
        "transform": "300px",
        "quality": "170px",
        "load": "150px",
        "opt_storage": "150px",
        "opt_code": "200px",
    })
    .tab_options(
        table_font_size="13px",
        heading_title_font_size="20px",
        heading_subtitle_font_size="13px",
        column_labels_font_weight="bold",
        column_labels_background_color="#1f2d3d",
        column_labels_font_size="13px",
        row_group_font_weight="bold",
        table_border_top_color="#1f2d3d",
        heading_background_color="#f4f6f8",
        stub_background_color="#eef1f5",
        data_row_padding="10px",
    )
    .opt_table_font(font="Arial")
    .tab_style(
        style=style.text(weight="bold", color="#1f2d3d"),
        locations=loc.stub(),
    )
    .tab_style(
        style=style.fill(color="#fbfcfd"),
        locations=loc.body(columns=["opt_storage", "opt_code"]),
    )
    .tab_source_note(source_note=md("*SPJ = Storage-Partitioned Join. SCD2 = Slowly Changing Dimension Type 2.*"))
)

# Display in a notebook:
gt

# Or export:
# gt.save("etl_table.png", scale=2)
# gt.write_raw_html("etl_table.html")

In [ ]:
gt.write_raw_html("etl_table.html")